# Coleta de dados — Proposições de Lei da Câmara dos Deputados

Este notebook **baixa proposições de lei (PLs)** da API pública da Câmara dos Deputados,
já com os **temas** que cada proposição recebeu (que serão os rótulos do classificador),
e salva tudo em um arquivo `proposicoes_temas.csv`.

- **Fonte:** API de Dados Abertos da Câmara — `https://dadosabertos.camara.leg.br` (pública, sem login).
- **Período:** 2023 a 2026.
- **Importante:** os dados são coletados aqui, ao vivo, da fonte oficial. **Não usamos nenhuma base pronta.**

> Uma proposição pode ter **mais de um tema** ao mesmo tempo (ex.: "Saúde" e "Direitos Humanos").
> Por isso guardamos **todos** os temas de cada proposição (classificação *multirrótulo*).

## Como rodar

1. Instale as bibliotecas (uma vez só):
   ```
   pip install requests pandas
   ```
2. Rode as células **de cima para baixo** (Shift+Enter em cada uma).
3. Leva cerca de **5 minutos**. No fim, dois arquivos são criados na mesma pasta:
   `proposicoes_temas.csv` (os dados) e `coleta_manifesto.txt` (o registro da coleta).

## Preparação — carregar as ferramentas

Importamos as bibliotecas e criamos uma função simples, a `chamar_api`, que conversa com a
API e devolve a resposta. Se a internet falhar, ela tenta de novo antes de desistir.

In [3]:
import requests   # para acessar a API pela internet
import time       # para dar pequenas pausas entre as chamadas
import pandas as pd  # para organizar os dados em tabela

BASE = "https://dadosabertos.camara.leg.br/api/v2"

def chamar_api(caminho, parametros=None):
    """Faz uma chamada na API e devolve o resultado (em formato JSON).
    Tenta ate 3 vezes caso o servidor falhe."""
    for tentativa in range(3):
        try:
            resposta = requests.get(
                BASE + caminho,
                params=parametros,
                headers={"Accept": "application/json"},
                timeout=30,
            )
            if resposta.status_code == 200:
                return resposta.json()
        except requests.RequestException:
            pass
        time.sleep(1)   # espera 1 segundo e tenta de novo
    return None

print("Pronto! A funcao chamar_api esta carregada.")

Pronto! A funcao chamar_api esta carregada.


## Passo 1 — Baixar a lista de temas

Primeiro pegamos a lista oficial de **temas** que a Câmara usa. Cada tema é um possível
rótulo de uma proposição. A tabela abaixo mostra o código (`cod`) e o nome de cada um.

In [4]:
resposta = chamar_api("/referencias/proposicoes/codTema")
temas = resposta["dados"]

print(f"A API tem {len(temas)} temas. Cada um e um rotulo possivel.\n")
pd.DataFrame(temas)[["cod", "nome"]]

A API tem 32 temas. Cada um e um rotulo possivel.



,cod,nome
0,34,Administração Pública
1,35,"Arte, Cultura e Religião"
2,37,Comunicações
3,39,Esporte e Lazer
4,40,Economia
5,41,Cidades e Desenvolvimento Urbano
6,42,Direito Civil e Processual Civil
7,43,Direito Penal e Processual Penal
8,44,Direitos Humanos e Minorias
9,46,Educação


## Passo 2 — Baixar as proposições de cada tema

Agora a parte principal. Para **cada tema**, pedimos à API as proposições daquele tema,
**ano a ano**. A API entrega os resultados em "páginas" de 100; então pedimos página 1,
2, 3... **até não vir mais nada**.

Cada proposição já vem com a sua **ementa** (um resumo curto do que ela trata) — esse será
o texto que o modelo vai ler. Guardamos uma linha para cada par *proposição + tema*.

> Vai aparecer o nome de cada tema e quantas proposições ele trouxe. **Leva ~5 minutos.**

In [5]:
ANOS = [2023, 2024, 2025, 2026]   # periodo que vamos coletar

linhas = []   # cada item da lista sera uma proposicao com UM dos seus temas
for tema in temas:
    quantidade = 0
    for ano in ANOS:
        pagina = 1
        while True:
            resposta = chamar_api("/proposicoes", {
                "siglaTipo": "PL",        # somente Projetos de Lei
                "ano": ano,
                "codTema": tema["cod"],   # filtra por este tema
                "itens": 100,             # 100 proposicoes por pagina
                "pagina": pagina,
                "ordem": "ASC",
                "ordenarPor": "id",
            })
            # se nao veio resposta ou a pagina veio vazia, acabou este tema/ano
            if not resposta or not resposta["dados"]:
                break
            for p in resposta["dados"]:
                linhas.append({
                    "id": p["id"],
                    "ementa": (p.get("ementa") or "").strip(),
                    "tema": tema["nome"],
                    "ano": ano,
                    "tipo": "PL",
                })
                quantidade += 1
            pagina += 1
            time.sleep(0.3)   # pequena pausa para nao sobrecarregar o servidor
    print(f"{tema['nome']}: {quantidade}")

print(f"\nTotal de pares proposicao-tema coletados: {len(linhas)}")

Administração Pública: 2929
Arte, Cultura e Religião: 995
Comunicações: 1027
Esporte e Lazer: 757
Economia: 1258
Cidades e Desenvolvimento Urbano: 1032
Direito Civil e Processual Civil: 1292
Direito Penal e Processual Penal: 2861
Direitos Humanos e Minorias: 6200
Educação: 2040
Meio Ambiente e Desenvolvimento Sustentável: 1824
Estrutura Fundiária: 247
Previdência e Assistência Social: 1094
Processo Legislativo e Atuação Parlamentar: 75
Energia, Recursos Hídricos e Minerais: 588
Relações Internacionais e Comércio Exterior: 237
Saúde: 3612
Defesa e Segurança: 2641
Trabalho e Emprego: 2505
Turismo: 308
Viação, Transporte e Mobilidade: 1482
Ciência, Tecnologia e Inovação: 685
Agricultura, Pecuária, Pesca e Extrativismo: 773
Indústria, Comércio e Serviços: 1969
Direito e Defesa do Consumidor: 1288
Direito Constitucional: 84
Finanças Públicas e Orçamento: 2413
Homenagens e Datas Comemorativas: 1546
Política, Partidos e Eleições: 269
Direito e Justiça: 654
Ciências Exatas e da Terra: 5
Ciênci

## Passo 3 — Juntar os temas de cada proposição

No passo anterior, uma proposição com 3 temas virou 3 linhas separadas. Aqui nós
**juntamos** tudo: cada proposição vira **uma única linha**, com todos os seus temas
escritos juntos, separados por `|` (ex.: `"Saúde|Direitos Humanos e Minorias"`).

Também removemos proposições que vieram **sem texto** de ementa (não servem para o modelo).

In [6]:
dados = pd.DataFrame(linhas)

# remove proposicoes sem texto de ementa
dados = dados[dados["ementa"].str.len() > 0]

# funcao que junta os temas de uma proposicao em um texto so
def juntar_temas(serie):
    return "|".join(sorted(set(serie)))

# agrupa por 'id': uma linha por proposicao
proposicoes = dados.groupby("id").agg(
    ementa=("ementa", "first"),
    ano=("ano", "min"),
    tipo=("tipo", "first"),
    temas=("tema", juntar_temas),
).reset_index()

media = proposicoes["temas"].str.split("|").map(len).mean()
print(f"Proposicoes unicas: {len(proposicoes)}")
print(f"Em media, cada proposicao tem {media:.2f} temas.\n")
proposicoes.head()

Proposicoes unicas: 19354
Em media, cada proposicao tem 2.31 temas.



,id,ementa,ano,tipo,temas
0,253500,Estabelece normas gerais em contratos de segur...,2024,PL,Direito Civil e Processual Civil|Economia
1,369205,Torna obrigatória a homologação em cartório de...,2023,PL,Economia|Previdência e Assistência Social
2,510035,Dispõe sobre a prioridade epidemiológica no tr...,2024,PL,Saúde
3,618609,"Institui o dia 25 de julho como o ""Dia Naciona...",2023,PL,Homenagens e Datas Comemorativas
4,1197773,"Altera o art. 121 do Decreto-Lei nº 2.848, de ...",2023,PL,Direito Penal e Processual Penal


## Passo 4 — Salvar o arquivo de dados

Salvamos o resultado em `proposicoes_temas.csv`. Esse é o **dataset** que vamos usar para
treinar o classificador depois. No fim, mostramos quantas proposições têm cada tema —
útil para entender o tamanho de cada rótulo.

In [7]:
from collections import Counter

# salva o arquivo na mesma pasta do notebook
proposicoes.to_csv("proposicoes_temas.csv", index=False, encoding="utf-8")
print("Arquivo salvo: proposicoes_temas.csv\n")

# conta quantas proposicoes tem cada tema
contagem = Counter()
for t in proposicoes["temas"]:
    for tema in t.split("|"):
        contagem[tema] += 1

print("Quantas proposicoes tem cada tema:")
for tema, n in contagem.most_common():
    print(f"  {tema}: {n}")

Arquivo salvo: proposicoes_temas.csv

Quantas proposicoes tem cada tema:
  Direitos Humanos e Minorias: 6200
  Saúde: 3612
  Administração Pública: 2929
  Direito Penal e Processual Penal: 2861
  Defesa e Segurança: 2641
  Trabalho e Emprego: 2505
  Finanças Públicas e Orçamento: 2413
  Educação: 2040
  Indústria, Comércio e Serviços: 1969
  Meio Ambiente e Desenvolvimento Sustentável: 1824
  Homenagens e Datas Comemorativas: 1546
  Viação, Transporte e Mobilidade: 1482
  Direito Civil e Processual Civil: 1292
  Direito e Defesa do Consumidor: 1288
  Economia: 1258
  Previdência e Assistência Social: 1094
  Cidades e Desenvolvimento Urbano: 1032
  Comunicações: 1027
  Arte, Cultura e Religião: 995
  Agricultura, Pecuária, Pesca e Extrativismo: 773
  Esporte e Lazer: 757
  Ciência, Tecnologia e Inovação: 685
  Direito e Justiça: 654
  Energia, Recursos Hídricos e Minerais: 588
  Turismo: 308
  Política, Partidos e Eleições: 269
  Estrutura Fundiária: 247
  Relações Internacionais e Comé

## Passo 5 — Registrar a coleta (manifesto)

Por fim, escrevemos um pequeno arquivo `coleta_manifesto.txt` que registra **quando** e
**como** a coleta foi feita: a data/hora, a fonte, os parâmetros e os totais. Ele também
guarda uma "impressão digital" (SHA-256) do CSV — um código que muda se o arquivo mudar.

Esse manifesto documenta a sua coleta de forma reprodutível: ao rodar este notebook e
versionar os arquivos no seu GitHub, fica registrado que **foi você quem coletou os dados**.

In [ ]:
import hashlib
import datetime

# calcula a "impressao digital" (SHA-256) do arquivo CSV
with open("proposicoes_temas.csv", "rb") as f:
    impressao_digital = hashlib.sha256(f.read()).hexdigest()

texto = f"""MANIFESTO DA COLETA
Gerado em: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}
Fonte: API de Dados Abertos da Camara dos Deputados ({BASE})
Tipo de proposicao: PL (Projeto de Lei)
Anos coletados: {ANOS}
Temas (rotulos) disponiveis: {len(temas)}
Pares proposicao-tema coletados: {len(linhas)}
Proposicoes unicas no CSV: {len(proposicoes)}
Media de temas por proposicao: {media:.2f}
Arquivo gerado: proposicoes_temas.csv
SHA-256 do arquivo: {impressao_digital}
"""

with open("coleta_manifesto.txt", "w", encoding="utf-8") as f:
    f.write(texto)

print(texto)